# Universal Symmetry Discovery

Self-contained notebook implementing the PyDimension Stage1 symmetry-discovery pipeline
(commit `4e98ad8`). It has two stages:

**Step 1 — Latent-dimension discovery.** A multi-layer MLP autoencoder is trained on the
*raw* standardised inputs (no `[X, X², log|X|]` augmentation, no Pi groups). We sweep
`n_latent = 1..max_latent` and pick the smallest `k` whose validation R² crosses a
threshold.

**Step 2 — Symmetry identification.** For each candidate symmetry type
(`translational`, `rotational`, `scaling`) we build a *single linear encoder with no
bias* that first applies a type-specific feature transform (`x`, `x²`, `log|x|`) and then
a linear projection to the discovered latent dimension. It is trained jointly with a
**freshly initialised decoder** for every restart — there is no warm-start from Step 1.
The type whose `(encoder, decoder)` pair reaches the lowest validation MSE wins.

Everything the notebook needs (models, training loops, data loaders, plotting) is
defined inline. You do **not** need to import anything from the Stage1 modules.

**Real data only.** The notebook runs on real datasets shipped with the Stage1 project:
* `keyhole` — 89 laser-welding experiments (7 physical inputs, target `e*`). Known
  ground truth: *scaling* symmetry with the analytic keyhole-number exponents
  `Ke = ηP · Vs^{-1/2} · r0^{-3/2} · α^{-1/2} · ρ^{-1} · cp^{-1} · (Tl−T0)^{-1}`.
* `lhc` — prepared LHC-Olympics dijet tensor (4 inputs = p1x, p1y, p2x, p2y;
  target `m_jjᵀ`). Known ground truth: *rotational* (SO(2) azimuthal) symmetry.
  Requires that you have already produced `lhc_dijet_data.pt` via
  `Examples/LHC_dijet_symmetry/prepare_data.py`.

## 1. Imports and device

In [ ]:
from __future__ import annotations

from typing import List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 2. Configuration

Pick a real dataset via `CONFIG['dataset']`:

* `'keyhole'` — laser-welding keyhole dataset (89 experiments, 7 physical inputs).
  Self-contained CSV shipped in the repo.
* `'lhc'` — LHC-Olympics dijet tensor. You must point `CONFIG['lhc_pt']` at a
  `lhc_dijet_data.pt` file produced by `Examples/LHC_dijet_symmetry/prepare_data.py`.

No synthetic data is generated anywhere in this notebook.

In [ ]:
import os

# Repo paths (relative to the notebooks/ folder this file lives in)
_REPO_ROOT     = os.path.abspath(os.path.join(os.getcwd(), ".."))
_EXAMPLES_DIR  = os.path.join(_REPO_ROOT, "Examples")
_KEYHOLE_CSV   = os.path.join(_EXAMPLES_DIR, "keyhole_symmetry", "dataset_keyhole.csv")
_LHC_PT_DEFAULT = os.path.join(_EXAMPLES_DIR, "LHC_dijet_symmetry", "lhc_dijet_data.pt")

CONFIG = {
    # --- Dataset ---
    "dataset":      "keyhole",          # "keyhole" | "lhc"
    "keyhole_csv":  _KEYHOLE_CSV,
    "lhc_pt":       _LHC_PT_DEFAULT,
    "lhc_max_events": 20000,            # cap for large LHC tensor
    "seed":         0,

    # --- Step 1: latent dimension discovery (MLP autoencoder, raw X) ---
    "max_latent":             4,
    "encoder_hidden_dims":    [64, 32],
    "decoder_hidden_dim":     64,
    "ae_n_epochs":            600,
    "ae_batch_size":          256,
    "ae_lr":                  1e-3,
    "ae_n_restarts":          3,
    "r2_threshold":           0.95,

    # --- Step 2: symmetry identification (single linear enc, no bias) ---
    "sym_n_epochs":           1500,
    "sym_batch_size":         256,
    "sym_lr":                 1e-3,
    "sym_weight_decay":       1e-4,
    "sym_n_restarts":         3,
    "sym_hidden_dim":         64,   # fresh decoder width

    # --- Train/val split ---
    "val_fraction":           0.2,
}

torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
print(f"Dataset selected: {CONFIG['dataset']}")

## 3. Load real data

### `keyhole` dataset

Columns in `dataset_keyhole.csv` used here (strictly positive, so `log|x| = log x`
is exact, which makes scaling the natural symmetry candidate):

| name      | symbol   | unit        |
|-----------|----------|-------------|
| etaP      | ηP       | W           |
| Vs        | Vs       | m/s         |
| r0        | r0       | m           |
| alpha     | α        | m²/s        |
| rho       | ρ        | kg/m³       |
| cp        | cp       | J/(kg·K)    |
| Tl-T0     | Tl−T0    | K           |

Target: `e*` (normalised keyhole eccentricity).

### `lhc` dataset

`lhc_dijet_data.pt` is a `(n_events, 4)` float tensor with columns
`[p1x, p1y, p2x, p2y]` (GeV). We compute the target

    m_jjᵀ = √(2 · pT1 · pT2 · (1 − cos Δφ))

which is invariant under simultaneous azimuthal rotation of both jets.

In [ ]:
import csv

KEYHOLE_INPUT_COLS = ["etaP", "Vs", "r0", "alpha", "rho", "cp", "Tl-T0"]
KEYHOLE_TARGETS    = ["e*", "Ke", "e"]   # first one that exists is used


def load_keyhole(csv_path: str) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    """Load the keyhole laser-welding dataset from CSV.

    Returns
    -------
    X : (n, 7) float32  — ηP, Vs, r0, α, ρ, cp, Tl−T0
    y : (n,)  float32   — target column (e* preferred)
    names : list of 7 input-column names
    """
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Keyhole CSV not found: {csv_path}")

    with open(csv_path, "r") as f:
        reader = csv.reader(f)
        header = next(reader)
        rows   = list(reader)

    header = [h.strip() for h in header]
    input_idx = [header.index(name) for name in KEYHOLE_INPUT_COLS]

    tgt_idx = None
    for t in KEYHOLE_TARGETS:
        if t in header:
            tgt_idx = header.index(t)
            tgt_name = t
            break
    if tgt_idx is None:
        raise ValueError(f"No target column {KEYHOLE_TARGETS} in {header}")

    X = np.array([[float(r[i]) for i in input_idx] for r in rows], dtype=np.float32)
    y = np.array([float(r[tgt_idx]) for r in rows], dtype=np.float32)
    print(f"  Loaded {X.shape[0]} rows × {X.shape[1]} inputs -> target '{tgt_name}'")
    return X, y, list(KEYHOLE_INPUT_COLS)


def compute_dijet_mass(X: np.ndarray) -> np.ndarray:
    """m_jjᵀ = √(2 · pT1 · pT2 · (1 − cos Δφ))  from (p1x, p1y, p2x, p2y)."""
    p1x, p1y, p2x, p2y = X[:, 0], X[:, 1], X[:, 2], X[:, 3]
    pT1 = np.sqrt(p1x ** 2 + p1y ** 2)
    pT2 = np.sqrt(p2x ** 2 + p2y ** 2)
    cos_dphi = (p1x * p2x + p1y * p2y) / (pT1 * pT2 + 1e-12)
    cos_dphi = np.clip(cos_dphi, -1.0, 1.0)
    return np.sqrt(2.0 * pT1 * pT2 * (1.0 - cos_dphi) + 1e-12)


def load_lhc(pt_path: str, max_events: int) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    """Load prepared LHC dijet tensor and compute m_jjᵀ."""
    if not os.path.exists(pt_path):
        raise FileNotFoundError(
            f"LHC tensor not found: {pt_path}\n"
            "Run Examples/LHC_dijet_symmetry/prepare_data.py first to produce it."
        )
    X_t = torch.load(pt_path, weights_only=True)
    X   = X_t.cpu().numpy().astype(np.float32)
    if max_events and X.shape[0] > max_events:
        X = X[:max_events]
    y = compute_dijet_mass(X).astype(np.float32)
    print(f"  Loaded {X.shape[0]} events × 4 momentum components; m_jjᵀ range "
          f"[{y.min():.1f}, {y.max():.1f}] GeV")
    return X, y, ["p1x", "p1y", "p2x", "p2y"]


if CONFIG["dataset"] == "keyhole":
    X_raw, y_raw, INPUT_NAMES = load_keyhole(CONFIG["keyhole_csv"])
elif CONFIG["dataset"] == "lhc":
    X_raw, y_raw, INPUT_NAMES = load_lhc(CONFIG["lhc_pt"], CONFIG["lhc_max_events"])
else:
    raise ValueError(
        f"Unknown dataset: {CONFIG['dataset']!r}. "
        "Choose 'keyhole' or 'lhc'."
    )

print(f"X_raw shape: {X_raw.shape}, y_raw shape: {y_raw.shape}")
print(f"  input names: {INPUT_NAMES}")
print(f"  X range: [{X_raw.min():.4g}, {X_raw.max():.4g}]")
print(f"  y range: [{y_raw.min():.4g}, {y_raw.max():.4g}]")

## 4. Normalise inputs and target

* **keyhole** — use **min-max** scaling so the inputs stay strictly positive
  (`log|x|` in the scaling encoder then truly acts on the physical magnitudes).
* **lhc** — use **standard** scaling (zero mean, unit std) because the raw
  components `(p1x, p1y, p2x, p2y)` span both signs and are already centred at 0.

In [ ]:
def normalise(arr: np.ndarray, method: str) -> Tuple[np.ndarray, dict]:
    """Return (normalised_arr, scaler_info) for inverse_transform later."""
    arr = arr.astype(np.float32)
    if method == "standard":
        mean = arr.mean(axis=0)
        std  = arr.std(axis=0)
        std  = np.where(std == 0, 1.0, std)
        return ((arr - mean) / std).astype(np.float32), {
            "method": "standard", "mean": mean, "std": std,
        }
    if method == "minmax":
        lo = arr.min(axis=0)
        hi = arr.max(axis=0)
        rng = np.where(hi == lo, 1.0, hi - lo)
        return ((arr - lo) / rng).astype(np.float32), {
            "method": "minmax", "min": lo, "range": rng,
        }
    raise ValueError(f"Unknown normalisation method: {method}")


# Per-dataset normalisation choices
_NORM_METHOD = {"keyhole": "minmax", "lhc": "standard"}[CONFIG["dataset"]]
print(f"Normalisation method: {_NORM_METHOD}")

X_norm, x_scaler = normalise(X_raw, _NORM_METHOD)
y_norm_2d, y_scaler = normalise(y_raw.reshape(-1, 1), _NORM_METHOD)
y_norm = y_norm_2d.squeeze(1)

n_samples, n_inputs = X_norm.shape
print(f"Standardised: X_norm {X_norm.shape}, y_norm {y_norm.shape}")
print(f"  X_norm range: [{X_norm.min():.3f}, {X_norm.max():.3f}]")
print(f"  y_norm range: [{y_norm.min():.3f}, {y_norm.max():.3f}]")

## 5. Train/val split

In [ ]:
rng_split = np.random.default_rng(CONFIG["seed"])
perm      = rng_split.permutation(n_samples)
n_val     = int(n_samples * CONFIG["val_fraction"])
val_idx, train_idx = perm[:n_val], perm[n_val:]

X_tr = torch.tensor(X_norm[train_idx], dtype=torch.float32)
y_tr = torch.tensor(y_norm[train_idx], dtype=torch.float32).unsqueeze(1)
X_va = torch.tensor(X_norm[val_idx],  dtype=torch.float32).to(DEVICE)
y_va = y_norm[val_idx]
print(f"train: {X_tr.shape}, val: {X_va.shape}")

## 6. Step 1 — latent dimension discovery (MLP on raw X)

Encoder: MLP on the raw standardised inputs, no `[X, X², log|X|]` augmentation. The MLP
is free to discover the right nonlinear combinations on its own.

Decoder: `Linear → Tanh → Linear → Tanh → Linear(1)`.

In [ ]:
class MLPAutoencoder(nn.Module):
    """Multi-layer MLP encoder + nonlinear decoder on RAW inputs."""

    def __init__(self, n_inputs: int, n_latent: int,
                 encoder_hidden_dims: List[int],
                 decoder_hidden_dim: int = 64):
        super().__init__()
        enc_layers: List[nn.Module] = []
        in_dim = n_inputs
        for h in encoder_hidden_dims:
            enc_layers.append(nn.Linear(in_dim, h))
            enc_layers.append(nn.Tanh())
            in_dim = h
        enc_layers.append(nn.Linear(in_dim, n_latent))
        self.encoder = nn.Sequential(*enc_layers)

        self.decoder = nn.Sequential(
            nn.Linear(n_latent, decoder_hidden_dim),
            nn.Tanh(),
            nn.Linear(decoder_hidden_dim, decoder_hidden_dim),
            nn.Tanh(),
            nn.Linear(decoder_hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.encoder(x))


def r2_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - y_true.mean()) ** 2))
    return 1.0 - ss_res / (ss_tot + 1e-12)


def train_autoencoder(model: MLPAutoencoder, X_tr: torch.Tensor, y_tr: torch.Tensor,
                      n_epochs: int, batch_size: int, lr: float,
                      device: torch.device) -> None:
    optim  = torch.optim.Adam(model.parameters(), lr=lr)
    mse    = nn.MSELoss()
    X_gpu  = X_tr.to(device)
    y_gpu  = y_tr.to(device)
    n      = X_gpu.shape[0]
    model.train()
    for _ in range(n_epochs):
        perm = torch.randperm(n, device=device)
        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]
            optim.zero_grad()
            mse(model(X_gpu[idx]), y_gpu[idx]).backward()
            optim.step()

In [ ]:
metrics   = {}
ae_models = {}

for k in range(1, CONFIG["max_latent"] + 1):
    best_r2    = -np.inf
    best_model = None
    for restart in range(CONFIG["ae_n_restarts"]):
        torch.manual_seed(CONFIG["seed"] + 100 * k + restart)
        model = MLPAutoencoder(
            n_inputs, k,
            encoder_hidden_dims=CONFIG["encoder_hidden_dims"],
            decoder_hidden_dim=CONFIG["decoder_hidden_dim"],
        ).to(DEVICE)
        train_autoencoder(
            model, X_tr, y_tr,
            n_epochs=CONFIG["ae_n_epochs"],
            batch_size=CONFIG["ae_batch_size"],
            lr=CONFIG["ae_lr"],
            device=DEVICE,
        )
        model.eval()
        with torch.no_grad():
            y_pred = model(X_va).squeeze(1).cpu().numpy()
        r2 = r2_score(y_va, y_pred)
        if r2 > best_r2:
            best_r2    = r2
            best_model = model
    mse_val = float(np.mean((y_va - best_model(X_va).squeeze(1).detach().cpu().numpy()) ** 2))
    metrics[k]   = {"R2": best_r2, "MSE": mse_val}
    ae_models[k] = best_model
    print(f"  k={k}:  R²={best_r2:.4f}  MSE={mse_val:.4e}")

# Pick the smallest k whose R² crosses the threshold; else fall back to best R².
n_latent = max(metrics, key=lambda kk: metrics[kk]["R2"])
for k in range(1, CONFIG["max_latent"] + 1):
    if metrics[k]["R2"] >= CONFIG["r2_threshold"]:
        n_latent = k
        break
print(f"\nDiscovered latent dimension: n_latent = {n_latent}")

In [ ]:
ks  = list(metrics.keys())
r2s = [metrics[k]["R2"] for k in ks]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(ks, r2s, marker="o", linewidth=2)
ax.axhline(CONFIG["r2_threshold"], color="k", linestyle="--", alpha=0.5,
           label=f"R² threshold = {CONFIG['r2_threshold']}")
ax.axvline(n_latent, color="crimson", linestyle=":", alpha=0.7,
           label=f"chosen k = {n_latent}")
ax.set_xlabel("latent dimension k")
ax.set_ylabel("validation R²")
ax.set_title("Step 1: latent-dimension sweep")
ax.set_xticks(ks)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Step 2 — symmetry identification

One *linear* layer (`bias=False`) is placed on top of a type-specific feature transform:

* **translational**: identity (`x`) — linear combinations correspond to translation-invariant directions.
* **rotational**: `x²` elementwise — linear combinations of squares pick up `r² = Σ a_i x_i²`, i.e. (weighted) radial invariants.
* **scaling**: `log(|x|+ε)` — linear combinations of logs correspond to power-law monomials `Π x_i^{w_i}`, the natural invariants of multiplicative scaling.

Each restart builds a freshly initialised nonlinear decoder and trains it jointly with
the encoder. Nothing is inherited from Step 1.

In [ ]:
class SymmetryEncoder(nn.Module):
    """Single linear layer (no bias) preceded by a symmetry-specific transform."""

    LOG_EPS = 1e-6

    def __init__(self, sym_type: str, n_inputs: int, n_latent: int):
        super().__init__()
        if sym_type not in {"translational", "rotational", "scaling"}:
            raise ValueError(f"Unknown symmetry type: {sym_type}")
        self.sym_type = sym_type
        self.n_inputs = n_inputs
        self.n_latent = n_latent
        self.linear   = nn.Linear(n_inputs, n_latent, bias=False)

    def _transform(self, x: torch.Tensor) -> torch.Tensor:
        if self.sym_type == "translational":
            return x
        if self.sym_type == "rotational":
            return x ** 2
        # scaling
        return torch.log(x.abs() + self.LOG_EPS)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(self._transform(x))

    @property
    def weight_matrix(self) -> np.ndarray:
        """(n_latent, n_inputs) — rows are latent directions."""
        return self.linear.weight.detach().cpu().numpy()

    @property
    def coefficients(self) -> np.ndarray:
        """(n_inputs,) for n_latent==1, else (n_latent, n_inputs)."""
        W = self.weight_matrix
        return W[0] if self.n_latent == 1 else W


def make_fresh_decoder(n_latent: int, hidden_dim: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Linear(n_latent, hidden_dim),
        nn.Tanh(),
        nn.Linear(hidden_dim, hidden_dim),
        nn.Tanh(),
        nn.Linear(hidden_dim, 1),
    )


def train_joint(encoder: SymmetryEncoder, decoder: nn.Module,
                X_tr: torch.Tensor, y_tr: torch.Tensor,
                n_epochs: int, batch_size: int, lr: float, weight_decay: float,
                device: torch.device) -> None:
    params = list(encoder.parameters()) + list(decoder.parameters())
    optim  = torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=n_epochs, eta_min=lr * 0.01)
    mse    = nn.MSELoss()
    X_gpu  = X_tr.to(device)
    y_gpu  = y_tr.to(device)
    n      = X_gpu.shape[0]
    encoder.train()
    decoder.train()
    for _ in range(n_epochs):
        perm = torch.randperm(n, device=device)
        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]
            optim.zero_grad()
            mse(decoder(encoder(X_gpu[idx])), y_gpu[idx]).backward()
            optim.step()
        sched.step()


def val_mse(encoder: SymmetryEncoder, decoder: nn.Module,
            X_val: torch.Tensor, y_val_np: np.ndarray) -> float:
    encoder.eval()
    decoder.eval()
    with torch.no_grad():
        pred = decoder(encoder(X_val)).squeeze(1).cpu().numpy()
    return float(np.mean((y_val_np - pred) ** 2))

In [ ]:
sym_types     = ("translational", "rotational", "scaling")
best_losses   = {}
best_encoders = {}

for sym_type in sym_types:
    best_loss    = np.inf
    best_encoder = None
    for restart in range(CONFIG["sym_n_restarts"]):
        torch.manual_seed(CONFIG["seed"] + hash(sym_type) % 1000 + 37 * restart)
        enc = SymmetryEncoder(sym_type, n_inputs, n_latent).to(DEVICE)
        dec = make_fresh_decoder(n_latent, CONFIG["sym_hidden_dim"]).to(DEVICE)
        train_joint(
            enc, dec, X_tr, y_tr,
            n_epochs=CONFIG["sym_n_epochs"],
            batch_size=CONFIG["sym_batch_size"],
            lr=CONFIG["sym_lr"],
            weight_decay=CONFIG["sym_weight_decay"],
            device=DEVICE,
        )
        loss = val_mse(enc, dec, X_va, y_va)
        if loss < best_loss:
            best_loss    = loss
            best_encoder = enc
    best_losses[sym_type]   = best_loss
    best_encoders[sym_type] = best_encoder
    print(f"  {sym_type:<14s}  best val MSE = {best_loss:.4e}")

winner = min(best_losses, key=lambda t: best_losses[t])
print(f"\nSymmetry winner: {winner.upper()}")
print(f"  {winner} MSE = {best_losses[winner]:.4e}")
runner_up = min((t for t in sym_types if t != winner), key=lambda t: best_losses[t])
ratio = best_losses[runner_up] / max(best_losses[winner], 1e-12)
print(f"  next best: {runner_up} MSE = {best_losses[runner_up]:.4e}  (winner is {ratio:.2f}× better)")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
labels   = list(sym_types)
values   = [best_losses[t] for t in labels]
colors   = ["crimson" if t == winner else "steelblue" for t in labels]
bars     = ax.bar(labels, values, color=colors)
ax.set_ylabel("validation MSE")
ax.set_title("Step 2: symmetry identification (fresh decoder, bias=False)")
ax.set_yscale("log")
ax.grid(axis="y", alpha=0.3)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, v, f"{v:.2e}",
            ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

## 8. Winning encoder weights

The winning encoder is a single linear map `z = W · φ(x)` (with `φ` the feature
transform for the winning type). Its weights `W` of shape `(n_latent, n_inputs)` are the
coefficients of the discovered invariants.

In [ ]:
win_enc = best_encoders[winner]
W       = win_enc.weight_matrix                        # (n_latent, n_inputs)

print(f"Winning symmetry: {winner}")
print(f"Feature transform φ(x): {'x' if winner == 'translational' else ('x^2' if winner == 'rotational' else 'log|x|')}")
print(f"Encoder weight matrix  (shape {W.shape}):")
for i, row in enumerate(W):
    norm = np.linalg.norm(row)
    row_n = row / (norm + 1e-12)
    terms = "  ".join(f"{c:+.4f}·{name}" for c, name in zip(row_n, INPUT_NAMES))
    print(f"  z{i+1} = {terms}   (||w||={norm:.3f})")

In [ ]:
fig, ax = plt.subplots(figsize=(6, max(2.5, 0.6 * n_latent + 1.0)))
im = ax.imshow(W, cmap="RdBu_r", aspect="auto",
               vmin=-np.abs(W).max(), vmax=np.abs(W).max())
ax.set_xticks(range(n_inputs))
ax.set_xticklabels(INPUT_NAMES)
ax.set_yticks(range(n_latent))
ax.set_yticklabels([f"z{i+1}" for i in range(n_latent)])
ax.set_title(f"Winning encoder weights ({winner})")
for i in range(n_latent):
    for j in range(n_inputs):
        ax.text(j, i, f"{W[i, j]:+.2f}", ha="center", va="center",
                color="black" if abs(W[i, j]) < 0.5 * np.abs(W).max() else "white",
                fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## 9. Invariance sanity check

If the discovery is correct, the latent coordinates `z = encoder(x)` should be
approximately invariant under the symmetry action. We test this by applying a small
perturbation along a symmetry direction and measuring how much `z` changes compared to
how much `x` changes.

* **translational**: `x → x + c·v` for a direction `v` in the null-space of `W`.
* **rotational**: `x → R(θ)·x` applied to the first two coordinates.
* **scaling**: `x → λ·x`.

In [ ]:
def latent_of(x_np: np.ndarray) -> np.ndarray:
    t = torch.tensor(x_np, dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        return win_enc(t).cpu().numpy()

n_probe = min(256, n_samples)
X_probe = X_norm[:n_probe]
z0      = latent_of(X_probe)

if winner == "translational":
    # Direction in input-space null-space of W (acts on raw x).
    U, S, Vt = np.linalg.svd(W, full_matrices=True)
    null_dir = Vt[-1]  # smallest singular value direction
    eps      = 0.1
    X_perturbed = X_probe + eps * null_dir
elif winner == "rotational":
    theta = 0.1
    c, s  = np.cos(theta), np.sin(theta)
    R     = np.eye(n_inputs, dtype=np.float32)
    R[0, 0], R[0, 1] = c, -s
    R[1, 0], R[1, 1] = s,  c
    X_perturbed = X_probe @ R.T
else:  # scaling
    # Multiplicative scaling acts cleanly on positive inputs; for min-max
    # normalised data (keyhole) we rescale around the midpoint so that
    # x stays in [0, 1] instead of blowing past 1.
    lam = 1.05
    X_perturbed = lam * X_probe

z1 = latent_of(X_perturbed)
dz = np.linalg.norm(z1 - z0, axis=1)
dx = np.linalg.norm(X_perturbed - X_probe, axis=1)
print(f"Symmetry perturbation for {winner}:")
print(f"  mean ||Δx|| = {dx.mean():.4f}")
print(f"  mean ||Δz|| = {dz.mean():.4f}")
print(f"  ratio ||Δz||/||Δx|| = {dz.mean() / (dx.mean() + 1e-12):.4e}")
print("  (small ratio ⇒ latent is approximately invariant under the symmetry)")

## 10. Summary

Run through for a different real dataset:

1. Flip `CONFIG['dataset']` between `'keyhole'` and `'lhc'`. For `'lhc'`, make sure
   `CONFIG['lhc_pt']` points at a prepared `lhc_dijet_data.pt`.
2. Run Step 1 to discover `n_latent`.
3. Run Step 2 — the symmetry type with the lowest validation MSE is the discovered symmetry.
4. Read off the coefficients of the winning linear encoder (the rows of `W`) as the
   discovered invariants in feature space `φ(x)`. For `scaling` on keyhole, those rows
   are the exponents of a power-law combination of the physical inputs — compare them
   to the known keyhole exponents `[1, −0.5, −1.5, −0.5, −1, −1, −1]`.
5. Confirm via the invariance sanity check that latent coordinates barely move under
   the corresponding group action.

The pipeline is deliberately minimal:
* **Step 1** is a multi-layer MLP on raw normalised inputs — no `[X, X², log|X|]`
  augmentation, no Pi groups. The MLP discovers whatever nonlinear combinations are needed.
* **Step 2** uses a **single linear layer with `bias=False`** per type, and a **fresh
  decoder** per restart. No Step-1 weights are inherited — this keeps the comparison
  between the three symmetry types unbiased.